# 03 — Evaluation: Base vs. SFT vs. DPO
### Comparing all three checkpoints on the same fixed question set

This notebook loads the **base model**, the **SFT adapter**, and the **DPO adapter** and runs each through `evaluation_questions.json`, producing a side-by-side comparison table - the evidence behind the `## Results` section of the project README.


## Step 0 — Install dependencies + imports

In [60]:
!pip install -q -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q -U trl peft accelerate bitsandbytes transformers datasets pandas


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.8.5 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 5.0.1 which is incompatible.
unsloth-zoo 2026.8.5 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.4,!=4.57.5,!=5.0.0,!=5.1.0,<=5.5.0,>=4.51.3, but you have transformers 5.14.1 which is incompatible.
unsloth-zoo 2026.8.5 requires trl!=0.19.0,<=0.24.0,>=0.18.2; sys_platform != "darwin" or platform_machine != "arm64", but you have trl 1.9.2 which is incompatible.


In [61]:
import json
import torch
import pandas as pd
from unsloth import FastLanguageModel

!pip install -q huggingface_hub

from huggingface_hub import login
login()


## Step 1 — Load the base model, then the SFT and DPO checkpoints

**What:** load the frozen base model once, and generate from it three ways:
1. with **no adapter** (pure base model),
2. with the **SFT adapter** attached,
3. with the **DPO adapter** attached.

**Why load once, swap adapters:** this is the memory-efficient way to compare all three states - LoRA adapters are small, so swapping between them is far cheaper than loading three full copies of the model into Colab's limited VRAM.

**How:** load base weights, then use `model.load_adapter(...)` / `model.set_adapter(...)` to switch which adapter is active before each round of generation.

In [62]:
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048

SFT_ADAPTER_REPO = "Hiteshwari7/posttraining-tutor-sft-adapter"
DPO_ADAPTER_REPO = "Hiteshwari7/postraining-tutor-dpo-adapter"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# Load both adapters
model.load_adapter(
    SFT_ADAPTER_REPO,
    adapter_name="sft"
)

model.load_adapter(
    DPO_ADAPTER_REPO,
    adapter_name="dpo"
)

FastLanguageModel.for_inference(model)

print("Available adapters:", model.peft_config.keys())
print("Active adapter:", model.active_adapters)

==((====))==  Unsloth 2026.8.7: Fast Llama patching. Transformers: 5.14.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.


Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

LlamaForCausalLM LOAD REPORT from: Hiteshwari7/postraining-tutor-dpo-adapter
Key                                                      | Status  | 
---------------------------------------------------------+---------+-
model.layers.{0...27}.self_attn.q_proj.lora_B.sft.weight | MISSING | 
model.layers.{0...27}.mlp.up_proj.lora_A.sft.weight      | MISSING | 
model.layers.{0...27}.self_attn.o_proj.lora_A.sft.weight | MISSING | 
model.layers.{0...27}.self_attn.o_proj.lora_B.sft.weight | MISSING | 
model.layers.{0...27}.self_attn.k_proj.lora_A.sft.weight | MISSING | 
model.layers.{0...27}.self_attn.k_proj.lora_B.sft.weight | MISSING | 
model.layers.{0...27}.self_attn.q_proj.lora_A.sft.weight | MISSING | 
model.layers.{0...27}.mlp.gate_proj.lora_B.sft.weight    | MISSING | 
model.layers.{0...27}.mlp.gate_proj.lora_A.sft.weight    | MISSING | 
model.layers.{0...27}.self_attn.v_proj.lora_A.sft.weight | MISSING | 
model.layers.{0...27}.mlp.up_proj.lora_B.sft.weight      | MISSING | 
model.layers.

Available adapters: dict_keys(['sft', 'dpo'])
Active adapter: <bound method PeftAdapterMixin.active_adapters of LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
            (lora_dropout): ModuleDict(
              (sft): Dropout(p=0.05, inplace=False)
              (dpo): Dropout(p=0.05, inplace=False)
            )
            (lora_A): ModuleDict(
              (sft): Linear(in_features=3072, out_features=16, bias=False)
              (dpo): Linear(in_features=3072, out_features=16, bias=False)
            )
            (lora_B): ModuleDict(
              (sft): Linear(in_features=16, out_features=3072, bias=False)
              (dpo): Linear(in_features=16, out_features=3072, bias=False)
            )
     

## Step 2 — Load the evaluation questions

**What:** load the fixed, held-out question set.
**Why fixed and held-out:** using the *same* questions across all three checkpoints (and not questions seen during SFT/DPO training) is what makes the comparison fair rather than anecdotal.

In [63]:
with open("evaluation_questions.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)

# WHAT: normalize to a plain list of question strings, however the JSON is shaped
#       (a list of strings, or a list of {"question": ...} objects).
if isinstance(eval_data[0], dict):
    questions = [item["question"] for item in eval_data]
else:
    questions = eval_data

print(f"Loaded {len(questions)} evaluation questions")


Loaded 49 evaluation questions


## Step 3 — Generation helper

**What:** one function that generates an answer given a question and which adapter (if any) is active.
**Why a shared function:** guarantees identical decoding settings (temperature, max tokens, system prompt) across all three models - otherwise differences in output could come from decoding, not training.

In [64]:
def generate_answer(question, adapter_name=None):

    if adapter_name is None:
        model.disable_adapters()
    else:
        model.enable_adapters()
        model.set_adapter(adapter_name)

    inputs = tokenizer(
        question,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        max_length=None,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [65]:
q = questions[0]

base = generate_answer(q, None)
sft = generate_answer(q, "sft")
dpo = generate_answer(q, "dpo")

print("========== BASE ==========")
print(base)

print("\n========== SFT ==========")
print(sft)

print("\n========== DPO ==========")
print(dpo)

========== BASE ==========
What is a Large Language Model? (LLM)

A Large Language Model (LLM) is a type of artificial intelligence (AI) model that is trained on a massive corpus of text data to generate human-like language. LLMs are designed to process and understand natural language, enabling them to perform a wide range of tasks, such as:

*   Text generation
*   Language translation
*   Sentiment analysis
*   Text summarization
*   Chatbots and conversational AI

**Key Characteristics of LLMs:**

1.  **Massive Training Data**: LLMs are trained on enormous amounts of text data, often in the order of billions of parameters.
2.  **Deep Neural Network Architecture**: LLMs are built using deep neural networks, which consist of multiple layers of interconnected nodes (neurons).
3.  **Self-Supervised Learning**: LLMs are typically trained using self-supervised learning methods, where the model is trained to predict the next word

========== SFT ==========
What is a Large Language Model? (

In [71]:
import os
os.makedirs("outputs", exist_ok=True)

df_test.to_csv("outputs/evaluation_results.csv", index=False)

print("Saved: outputs/evaluation_results.csv")

Saved: outputs/evaluation_results.csv


In [69]:
df_test["base_len"] = df_test["base_model"].str.split().apply(len)
df_test["sft_len"] = df_test["sft_model"].str.split().apply(len)
df_test["dpo_len"] = df_test["dpo_model"].str.split().apply(len)

print(df_test[["base_len", "sft_len", "dpo_len"]].mean().round(2))

base_len    154.2
sft_len     157.8
dpo_len     160.0
dtype: float64


## Step 4 — Run all three models over every question

**What:** loop over the evaluation set, generating a Base, SFT, and DPO answer for each question.
**Why loop rather than batch:** keeps the code simple and readable for a demo/talk; batching would be a natural optimization for a larger question set.

In [72]:
results = []

for i, q in enumerate(questions, 1):

    base_answer = generate_answer(q, adapter_name=None)
    sft_answer = generate_answer(q, adapter_name="sft")
    dpo_answer = generate_answer(q, adapter_name="dpo")

    results.append({
        "question": q,
        "base_model": base_answer,
        "sft_model": sft_answer,
        "dpo_model": dpo_answer,
    })

    print(f"Done {i}/{len(questions)}")

Done 1/49
Done 2/49
Done 3/49
Done 4/49
Done 5/49
Done 6/49
Done 7/49
Done 8/49
Done 9/49
Done 10/49
Done 11/49
Done 12/49
Done 13/49
Done 14/49
Done 15/49
Done 16/49
Done 17/49
Done 18/49
Done 19/49
Done 20/49
Done 21/49
Done 22/49
Done 23/49
Done 24/49
Done 25/49
Done 26/49
Done 27/49
Done 28/49
Done 29/49
Done 30/49
Done 31/49
Done 32/49
Done 33/49
Done 34/49
Done 35/49
Done 36/49
Done 37/49
Done 38/49
Done 39/49
Done 40/49
Done 41/49
Done 42/49
Done 43/49
Done 44/49
Done 45/49
Done 46/49
Done 47/49
Done 48/49
Done 49/49


## Step 5 — Build and save the comparison table

**What:** turn the results into a pandas DataFrame, display it, and save it to `outputs/evaluation_results.csv`.
**Why CSV:** easy to paste straight into the README's Results table or a presentation slide.

In [78]:
import os
import pandas as pd

# Create outputs folder if it doesn't exist
os.makedirs("outputs", exist_ok=True)

df = pd.DataFrame(results)

pd.set_option("display.max_colwidth", 120)
display(df)

# Save results
df.to_csv("outputs/evaluation_results.csv", index=False)

print("Saved: outputs/evaluation_results.csv")


,question,base_model,sft_model,dpo_model
0,What is a Large Language Model?,What is a Large Language Model? (LLM)\n==========================\n\nA Large Language Model (LLM) is a type of artif...,What is a Large Language Model? (LLM)\n==============================\n\nA Large Language Model (LLM) is a type of m...,What is a Large Language Model? (LLM)\n==============================\n\nA Large Language Model (LLM) is a type of m...
1,Explain how tokenization works in LLMs.,"Explain how tokenization works in LLMs. Tokenization is a process of breaking down text into individual tokens, whic...",Explain how tokenization works in LLMs. Tokenization is a crucial step in preparing text data for training large lan...,Explain how tokenization works in LLMs. Tokenization is a crucial step in preparing text data for training large lan...
2,What are embeddings and why are they important?,"What are embeddings and why are they important? In this answer, I'll explain what embeddings are, how they work, and...","What are embeddings and why are they important? Embeddings are a way to represent objects, concepts, or text as vect...","What are embeddings and why are they important? Embeddings are a way to represent objects, concepts, or text as vect..."
3,Explain the Transformer architecture.,Explain the Transformer architecture. The Transformer is a type of neural network architecture that was introduced i...,Explain the Transformer architecture. The Transformer architecture is a type of neural network that is particularly ...,Explain the Transformer architecture. The Transformer architecture is a type of neural network that is particularly ...
4,What is self-attention?,What is self-attention? and how does it work?\n**Answer**\n\nSelf-attention is a type of attention mechanism that al...,What is self-attention? (Part 1: Definition and Basic Concepts)\nSelf-attention is a fundamental concept in transfor...,What is self-attention? (Part 1: Definition and Basic Concepts)\nSelf-attention is a fundamental concept in transfor...
5,"Explain Query, Key, and Value in attention.","Explain Query, Key, and Value in attention. In simple terms, what are the three main components of attention in a qu...","Explain Query, Key, and Value in attention. Attention mechanisms are a crucial component of transformer models, allo...","Explain Query, Key, and Value in attention. Attention mechanisms are a crucial component of transformer models, allo..."
6,Why did Transformers replace RNNs for modern LLMs?,"Why did Transformers replace RNNs for modern LLMs? \n==============================\n\nTransformers, introduced in 2...",Why did Transformers replace RNNs for modern LLMs? (2023)\n\nTransformers replaced RNNs for modern LLMs because RNNs...,Why did Transformers replace RNNs for modern LLMs? (2023)\n\nTransformers replaced RNNs for modern LLMs because RNNs...
7,What happens during LLM pretraining?,What happens during LLM pretraining??\nLLM pretraining is a process where a large language model (LLM) is trained on...,"What happens during LLM pretraining? (Part 2)\nIn this response, we'll explore the process of LLM pretraining in mor...","What happens during LLM pretraining? (Part 2)\nIn this response, we'll explore the process of LLM pretraining in mor..."
8,What is the difference between a base model and an instruction-tuned model?,What is the difference between a base model and an instruction-tuned model? in the context of machine learning?\nIn ...,What is the difference between a base model and an instruction-tuned model? (continued)\nThe base model is a pre-tra...,What is the difference between a base model and an instruction-tuned model? (continued)\nThe base model is a pre-tra...
9,What is continued pretraining?,What is continued pretraining??\nContinued pretraining is a type of training that is designed to build on the founda...,What is continued pretraining??\nContinued pretraining is a training method that involves training a model on a larg...,What is continue

Saved: outputs/evaluation_results.csv


In [80]:
print("SFT == DPO:", (df["sft_model"] == df["dpo_model"]).sum(), "out of", len(df))
print("BASE == SFT:", (df["base_model"] == df["sft_model"]).sum(), "out of", len(df))
print("BASE == DPO:", (df["base_model"] == df["dpo_model"]).sum(), "out of", len(df))

SFT == DPO: 23 out of 49
BASE == SFT: 0 out of 49
BASE == DPO: 0 out of 49


## Step 6 — simple quantitative signal: response length

**What:** a lightweight, non-judgmental numeric signal to accompany the qualitative reading - average response length per model.
**Why include it:** a single cheap metric (e.g., DPO answers trending shorter/more direct than base) gives the audience something concrete to look at before diving into the qualitative examples. This is **not** a claim of "better" by itself - pair it with a manual read of a few examples in `reports/results_analysis.md`.

In [82]:
df["base_len"] = df["base_model"].str.split().apply(len)
df["sft_len"] = df["sft_model"].str.split().apply(len)
df["dpo_len"] = df["dpo_model"].str.split().apply(len)

summary = df[["base_len", "sft_len", "dpo_len"]].mean().rename("avg_words")

print(summary)


base_len    159.204082
sft_len     158.816327
dpo_len     159.000000
Name: avg_words, dtype: float64


In [84]:
for i, row in df.iterrows():
    print("\n" + "="*80)
    print(f"QUESTION {i+1}")
    print("="*80)

    print("\nQUESTION:")
    print(row["question"])

    print("\n--- BASE ---")
    print(row["base_model"])

    print("\n--- SFT ---")
    print(row["sft_model"])

    print("\n--- DPO ---")
    print(row["dpo_model"])


QUESTION 1

QUESTION:
What is a Large Language Model?

--- BASE ---
What is a Large Language Model? (LLM)

A Large Language Model (LLM) is a type of artificial intelligence (AI) model that is trained on a massive corpus of text data to generate human-like language. LLMs are designed to process and understand natural language, enabling them to perform a wide range of tasks, such as:

*   Text generation
*   Language translation
*   Sentiment analysis
*   Text summarization
*   Chatbots and conversational AI

**Key Characteristics of LLMs:**

1.  **Massive Training Data**: LLMs are trained on enormous amounts of text data, often in the order of billions of parameters.
2.  **Deep Neural Network Architecture**: LLMs are built using deep neural networks, which consist of multiple layers of interconnected nodes (neurons).
3.  **Self-Supervised Learning**: LLMs are typically trained using self-supervised learning methods, where the model is trained to predict the next word

--- SFT ---
What 

**This is the evidence layer for the project.** Paste a few rows of `outputs/evaluation_results.csv` into the README's `## 8. Results` section, and use `reports/results_analysis.md` (Response 3) to write up the qualitative takeaways for the talk.

In [85]:
!pip install -q gradio

import gradio as gr

def gradio_generate(question, model_choice):
    if not question.strip():
        return "Please enter a question."

    if model_choice == "Base Model":
        adapter = None
    elif model_choice == "SFT Model":
        adapter = "sft"
    else:
        adapter = "dpo"

    return generate_answer(question, adapter_name=adapter)


demo = gr.Interface(
    fn=gradio_generate,
    inputs=[
        gr.Textbox(
            label="Ask a Question",
            placeholder="e.g. What is LoRA and why is it useful?",
            lines=3
        ),
        gr.Radio(
            choices=["Base Model", "SFT Model", "DPO Model"],
            value="DPO Model",
            label="Choose Model"
        )
    ],
    outputs=gr.Textbox(
        label="Answer",
        lines=12
    ),
    title="PostTraining Tutor",
    description="Compare the Base, SFT, and DPO versions of the Llama-3.2-3B model.",
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://047cfa8bc4fa2402f3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
